# Imports and functions

In [1]:
# functions

from ensemble_chaos_tools import check_chaos
from earthkit.regrid import interpolate

import xarray as xr
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.ticker as mtickers
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
from matplotlib import rcParams
import cartopy.crs as ccrs
import nicopal as ncp
import json

import metpy.calc as mpcalc
from metpy.interpolate import interpolate_to_grid
from metpy.units import units
from pyextremes import EVA
from colindex2 import Detect

from tqdm.notebook import tqdm
import os
import glob
import pickle
import math
import shutil
import json

%load_ext autoreload
%autoreload 2
# %config InlineBackend.figure_format="jpeg"

In [2]:
def setup_latex_style(base_size=10):
    rcParams.update(
        {
            "font.size": base_size,  # Base size for everything
            "axes.titlesize": base_size,  # slightly larger for (a), (b), etc.
            "axes.labelsize": base_size - 1,  # Axis and colorbar labels
            "xtick.labelsize": base_size - 1,  # Slightly smaller for ticks
            "ytick.labelsize": base_size - 1,
            "legend.fontsize": base_size - 1,
            "font.family": "serif",
            "font.serif": ["cmr10"],
            "mathtext.fontset": "cm",
            "axes.formatter.use_mathtext": True,  # Fixes minus signs
            "contour.linewidth": 0.6,
        }
    )


def get_figsize(width_pt, fraction=1.0, height_factor=0.75):
    inches_per_pt = 1.0 / 72.27
    fig_width_in = width_pt * fraction * inches_per_pt
    fig_height_in = fig_width_in * height_factor
    return (fig_width_in, fig_height_in)


setup_latex_style()

WIDTH_PAPER = 372.0
WIDTH_INSA = 496.0

N320 grid coordinates

In [3]:
with open("/homedata/pchevali/n320_coordinates.pkl", "rb") as f:
    grid = pickle.load(f)

Some useful functions

In [4]:
def get_nearest_point(ds, target_lat, target_lon):
    distances = (ds["latitude"] - target_lat) ** 2 + (ds["longitude"] - target_lon) ** 2
    return distances.argmin().compute().item()

Figure out the mask for outputs of aifs

In [5]:
mask = (
    (grid["latitude"] > 15)
    * (grid["latitude"] < 75)
    * ((grid["longitude"] > 300) + (grid["longitude"] < 30))
)
with open("/homedata/pchevali/mask_eu.pkl", "wb") as f:
    pickle.dump(mask, f)

Import AIFS Simulations

In [6]:
z = xr.open_dataset(
    "/scratchx/pchevali/AIFS_OUTPUTS_REGRIDDED/2019071300-boosting_2019_paris_record/aifs_ensemble-2019071300-boosting_2019_paris_record-z.nc",
)

In [7]:
def open_iterative_boosting(variables, n_phases, base_dir, exp_name):
    phases = range(1, n_phases + 1)
    datasets_results = {}
    for phase in phases:
        datasets_results[phase] = {}
        for var in variables:
            pattern = f"{
                base_dir}/{exp_name}/iterative_boosting_phase{phase}-{exp_name}-{var}.nc"
            file_paths = glob.glob(pattern)
            ds = xr.open_dataset(
                file_paths[0],
                decode_timedelta=True,
                chunks="auto",
                engine="h5netcdf",
            )
            ds = ds.assign_coords(longitude=(((ds.longitude + 180) % 360) - 180))
            datasets_results[phase][var] = ds[var]
    return datasets_results


def reconstruct_last_phase_trajectories(dataset_results):
    # N° of phases
    N_PHASES = len(dataset_results)

    # map of the parents
    parent_map = {}
    for phase in dataset_results:
        ds = dataset_results[phase]["2t"]
        for m, p in zip(ds.member_id.values, ds.parent_id.values):
            parent_map[m] = p

    # dict {var : xarray.dataset}
    final_trajectories = {}

    # ids of the member of the last phase
    last_phase_members = dataset_results[N_PHASES]["2t"].member_id.values

    # for every variable
    for var in dataset_results[1].keys():

        # array of xarray.dataarray, one per member in last phase
        final_trajectories[var] = []

        # for every member in the last phase, trace back to the beginning and combine overriding with data from the older phase always
        # then set time to the original parent's starttime and remake the step variable (coordinate)
        for curr_id in last_phase_members:
            # actual number of this member
            id_in_res = int(curr_id.split("_m")[1])
            # trajectory we're rebuilding (one member for one variable)
            trajectory = None

            # for every phase (starting from the oldest) (ex 3,2,1)
            for phase in reversed(range(1, N_PHASES + 1)):
                # extract the data for the right member
                ds_phase_var_member = dataset_results[phase][var]
                idx_curr_id = np.where(ds_phase_var_member.member_id.values == curr_id)[
                    0
                ][0]
                ds_phase_var_member = ds_phase_var_member.isel(number=idx_curr_id)

                # put valid_time as the proper coordinate and remove step, time, parent_id, member_id
                ds_phase_var_member = ds_phase_var_member.swap_dims(
                    {"step": "valid_time"}
                ).drop_vars(["step", "time", "parent_id", "member_id"])
                # print(ds_phase_var_member)

                if trajectory is None:  # if we're at the last phase/first iter
                    trajectory = ds_phase_var_member
                else:  # if we're after
                    # smallest valid_time in the phase after this one
                    first_newer_time = trajectory.valid_time.min()
                    # we take only what's before that smallest valid time
                    ds_phase_var_member = ds_phase_var_member.where(
                        ds_phase_var_member.valid_time < first_newer_time, drop=True
                    )
                    # concat them in proper order
                    trajectory = xr.concat(
                        [ds_phase_var_member, trajectory], dim="valid_time"
                    )
                # change current id for next phase (parent id)
                curr_id = parent_map[curr_id]
                # if end of iteration format the dataarray
                if phase == 1:
                    trajectory["time"] = pd.to_datetime(curr_id, format="%Y%m%d%H")
                    trajectory["step"] = trajectory.valid_time - trajectory.time
                    trajectory = trajectory.swap_dims({"valid_time": "step"})
                    trajectory["number"] = id_in_res
            # append to trajectories
            final_trajectories[var].append(trajectory)
        # merge by number dimension
        final_trajectories[var] = xr.concat(final_trajectories[var], dim="number")
    return xr.merge(final_trajectories.values())

In [33]:
variables = [
    "2t",
    "2d",
    "sp",
    # "tp",
    "t_250",
    "t_300",
    "t_400",
    "t_500",
    "t_600",
    "t_700",
    "t_850",
    "t_925",
    "t_1000",
    "u_250",
    "u_300",
    "u_400",
    "u_500",
    "u_600",
    "u_700",
    "u_850",
    "u_925",
    "u_1000",
    "v_250",
    "v_300",
    "v_400",
    "v_500",
    "v_600",
    "v_700",
    "v_850",
    "v_925",
    "v_1000",
    "w_250",
    "w_300",
    "w_400",
    "w_500",
    "w_600",
    "w_700",
    "w_850",
    "w_925",
    "w_1000",
    "z_500",
]
base_dir = "/scratchx/pchevali/ITERATIVE_BOOSTING_PROCESSED"
exp_name = "return_levels_JJA_run1"
datasets_results_R1 = open_iterative_boosting(variables, 3, base_dir, exp_name)
dataset_results_reconstructed_R1 = reconstruct_last_phase_trajectories(datasets_results_R1)

In [11]:
variables = [
    "2t",
]
base_dir = "/scratchx/pchevali/ITERATIVE_BOOSTING_PROCESSED"
exp_name = "return_levels_JJA_run2"
datasets_results_R2 = open_iterative_boosting(variables, 3, base_dir, exp_name)
dataset_results_reconstructed_R2 = reconstruct_last_phase_trajectories(datasets_results_R2)

Indexing the latitude and longitude from the AIFS runs (and some common points and zones)

In [12]:
LON = dataset_results_reconstructed_R1.longitude.values
LAT = dataset_results_reconstructed_R1.latitude.values

In [13]:
paris_latitude = 49
paris_longitude = 2.5

plot_region_latitude_bnd = slice(23, 67)
plot_region_longitude_bnd = slice(-30, 40)

average_latitude_bnd = slice(44.5, 55)
average_longitude_bnd = slice(-5, 5.5)

cutoff_region_lat_bnd = slice(-13, -7)
cutoff_region_lon_bnd = slice(41, 43)

anticyclone_center_region_lat_bnd = slice(48, 50)
anticyclone_center_region_lon_bnd = slice(0, 6)

average_region = (
    (LAT >= average_latitude_bnd.start)
    & (LAT <= average_latitude_bnd.stop)
    & (LON >= average_longitude_bnd.start)
    & (LON <= average_longitude_bnd.stop)
)

plot_region_indexs = (
    (LAT >= plot_region_latitude_bnd.start)
    & (LAT <= plot_region_latitude_bnd.stop)
    & (LON >= plot_region_longitude_bnd.start)
    & (LON <= plot_region_longitude_bnd.stop)
)

cutoff_region_indexs = (
    (LAT >= cutoff_region_lat_bnd.start)
    & (LAT <= cutoff_region_lat_bnd.stop)
    & (LON >= cutoff_region_lon_bnd.start)
    & (LON <= cutoff_region_lon_bnd.stop)
)

anticyclone_center_region_indexs = (
    (LAT >= anticyclone_center_region_lat_bnd.start)
    & (LAT <= anticyclone_center_region_lat_bnd.stop)
    & (LON >= anticyclone_center_region_lon_bnd.start)
    & (LON <= anticyclone_center_region_lon_bnd.stop)
)

anticyclone_search_lat_bnd = slice(35, 60)
anticyclone_search_lon_bnd = slice(-25, 20)

anticyclone_search_region_indexs = (
    (LAT >= anticyclone_search_lat_bnd.start)
    & (LAT <= anticyclone_search_lat_bnd.stop)
    & (LON >= anticyclone_search_lon_bnd.start)
    & (LON <= anticyclone_search_lon_bnd.stop)
)

paris_big_box_lon = slice(0, 5)
paris_big_box_lat = slice(46, 50)
paris_big_box_region = (
    (LAT >= paris_big_box_lat.start)
    & (LAT <= paris_big_box_lat.stop)
    & (LON >= paris_big_box_lon.start)
    & (LON <= paris_big_box_lon.stop)
)

paris_index = get_nearest_point(
    datasets_results_R1[1]["2t"], paris_latitude, paris_longitude
)

ERA5 climatology

In [14]:
era5_paris_ts = xr.open_dataset(
    "reanalysis-era5-single-levels-timeseries-sfcq047uaxj.nc"
)
max_paris_era5_00utc = (
    era5_paris_ts.t2m.sel(valid_time=era5_paris_ts.valid_time.dt.hour.isin([0]))
    .max()
    .item()
    - 273.15
)
max_paris_era5_12utc = (
    era5_paris_ts.t2m.sel(valid_time=era5_paris_ts.valid_time.dt.hour.isin([12]))
    .max()
    .item()
    - 273.15
)
max_paris_era5_18utc = (
    era5_paris_ts.t2m.sel(valid_time=era5_paris_ts.valid_time.dt.hour.isin([18]))
    .max()
    .item()
    - 273.15
)
print(max_paris_era5_00utc, max_paris_era5_12utc, max_paris_era5_18utc)

26.865930175781273 39.82009277343752 37.7866149902344


In [15]:
# patterns = [
#     "/bdd/ERA5/NETCDF/GLOBAL_025/hourly/AN_SF/*/t2m.*06.as1e5.GLOBAL_025.nc",
#     "/bdd/ERA5/NETCDF/GLOBAL_025/hourly/AN_SF/*/t2m.*07.as1e5.GLOBAL_025.nc",
#     "/bdd/ERA5/NETCDF/GLOBAL_025/hourly/AN_SF/*/t2m.*08.as1e5.GLOBAL_025.nc",
# ]

# paths_t2m_for_climato = []
# for pattern in patterns:
#     files = glob.glob(pattern)
#     for file in files:
#         year_val = int(file.split("/")[7])
#         if year_val > 1990:
#             paths_t2m_for_climato.append(file)

# era5_t2m_for_climato = xr.open_mfdataset(
#     paths_t2m_for_climato, parallel=True, engine="h5netcdf"
# )

# era5_t2m_for_climato = era5_t2m_for_climato.sel(
#     time=era5_t2m_for_climato.time.dt.hour.isin([0, 6, 12, 18])
# )

# era5_2t_climato = era5_t2m_for_climato.groupby("time.hour").mean("time")

# era5_2t_climato.to_netcdf(
#     "/homedata/pchevali/ERA5_CLIMATO/t2m_jja_era5_climato.nc", engine="h5netcdf"
# )

# patterns = [
#     "/bdd/ERA5/NETCDF/GLOBAL_025/hourly/AN_PL/*/ta.*06.ap1e5.GLOBAL_025.nc",
#     "/bdd/ERA5/NETCDF/GLOBAL_025/hourly/AN_PL/*/ta.*07.ap1e5.GLOBAL_025.nc",
#     "/bdd/ERA5/NETCDF/GLOBAL_025/hourly/AN_PL/*/ta.*08.ap1e5.GLOBAL_025.nc",
# ]

# paths_ta_for_climato = []
# for pattern in patterns:
#     files = glob.glob(pattern)
#     for file in files:
#         year_val = int(file.split("/")[7])
#         if year_val > 1990:
#             paths_ta_for_climato.append(file)

# era5_ta_for_climato = xr.open_mfdataset(
#     paths_ta_for_climato, parallel=True, engine="h5netcdf"
# )

# era5_ta_for_climato = era5_ta_for_climato.sel(
#     time=era5_ta_for_climato.time.dt.hour.isin([0, 6, 12, 18]),level=[250,300,400,500,600,700,850,925,1000]
# )


# era5_ta_climato = era5_ta_for_climato.groupby(["time.hour"]).mean("time")

# era5_ta_climato.to_netcdf(
#     "/homedata/pchevali/ERA5_CLIMATO/ta_jja_era5_climato.nc", engine="h5netcdf"
# )
# print("DONE!")

In [16]:
climato_era5 = xr.open_dataset(
    "/homedata/pchevali/ERA5_CLIMATO/t2m_jja_era5_climato.nc"
)
climato_era5 = climato_era5.isel(hour=climato_era5.hour.isin([12]))
climato_era5 = interpolate(
    climato_era5["t2m"].values, {"grid": [0.25, 0.25]}, {"grid": "N320"}
)
climato_era5 = xr.DataArray(
    data=climato_era5,
    dims=["values"],
    coords={
        "latitude": ("values", grid["latitude"]),
        "longitude": ("values", grid["longitude"]),
    },
)
climato_era5 = climato_era5.assign_coords(
    longitude=(((climato_era5.longitude + 180) % 360) - 180)
).values[mask]

In [17]:
climato_era5_ta = xr.open_dataset(
    "/homedata/pchevali/ERA5_CLIMATO/ta_jja_era5_climato.nc"
)

output_datasets = [] 

for hour in climato_era5_ta.hour.values:
    level_datasets = [] 
    for level in climato_era5_ta.level.values:
        interpolated_values = interpolate(
            climato_era5_ta.sel(hour=hour, level=level)["ta"].values, 
            in_grid={"grid": [0.25, 0.25]}, 
            out_grid={"grid": "N320"}
        )
        interpolated_da = xr.DataArray(
            interpolated_values,
            dims=["values"],
            coords={
                "latitude": ("values", grid["latitude"]),
                "longitude": ("values", grid["longitude"]),
                "hour": hour,
                "level": level
            }
        )
        level_datasets.append(interpolated_da) 
    output_datasets.append(level_datasets) 

climato_era5_ta_interpolated = xr.combine_nested(
    output_datasets, 
    concat_dim=["hour", "level"]
)

climato_era5_ta_interpolated = climato_era5_ta_interpolated.assign_coords(
    longitude=(((climato_era5_ta_interpolated.longitude + 180) % 360) - 180)
)

climato_era5_ta_interpolated = climato_era5_ta_interpolated.isel(values=mask)

Functions for the MSE

In [18]:
z = xr.open_dataset(
    "/scratchx/pchevali/z_cutout.nc",
)

In [19]:
def moist_static_energy(T_s, T_500, T_2d, sp, z_500):
    # constants
    c_p = 1004.0  # Specific heat of air at constant pressure (J/kg/K)
    L_v = 2.5e6  # Latent heat of vaporization (J/kg)
    epsilon = 0.622  # Molar ratio of water vapor to dry air

    def calc_vapor_pressure(T_kelvin):
        # tetens formula (gives kpa but we want pa)
        T_celsius = T_kelvin - 273.15
        e_hpa = 6.1094 * np.exp((17.625 * T_celsius) / (T_celsius + 243.04))
        return e_hpa * 100

    e_actual = calc_vapor_pressure(T_2d)
    q_s = epsilon * (e_actual / sp)
    MSE_s = (c_p * T_s) + (L_v * q_s) + z["z"]

    e_sat_500 = calc_vapor_pressure(T_500)
    q_sat_500 = epsilon * (e_sat_500 / 50000)
    MSE_500_star = (c_p * T_500) + (L_v * q_sat_500) + z_500

    return MSE_500_star, MSE_s


def MSE_compute_and_plot(dataset, region=average_region, title=None):

    #tp = (dataset["tp"] * 1000).squeeze()  # millimeters

    MSE_500_star, MSE_s = moist_static_energy(
        dataset["2t"], dataset["t_500"], dataset["2d"], dataset["sp"], dataset["z_500"]
    )
    x_axis = dataset["2t"].step.values / np.timedelta64(1, "D")

    mse_500_plot = (
        MSE_500_star.isel(values=region).mean("values").squeeze().compute() / 1000
    )
    mse_s_plot = MSE_s.isel(values=region).mean("values").squeeze().compute() / 1000
    #tp_plot = tp.isel(values=region).mean("values").squeeze().compute()
    T_s_plot = (
        dataset["2t"].isel(values=region).mean("values").squeeze().compute() - 273.15
    )

    fig, (ax1, ax3) = plt.subplots(
        2, 1, figsize=get_figsize(WIDTH_INSA,1,0.75), sharex=True, gridspec_kw={"height_ratios": [1.5, 1]}
    )

    #### MSE plot

    ax1.plot(x_axis, mse_s_plot, color="limegreen", label="$MSE_s$ (Surface)")
    ax1.plot(
        x_axis,
        mse_500_plot,
        color="darkgreen",
        label="$MSE_{500}^*$ (500hPa Saturation)",
    )
    ax1.fill_between(
        x_axis,
        mse_s_plot,
        mse_500_plot,
        where=(mse_s_plot > mse_500_plot),
        color="red",
        interpolate=True,
        alpha=0.1,
        label="Instability",
    )  # periods where mse_s-mse_500>0

    ax1.set_ylabel("Moist Static Energy (J/g)", fontsize="large")
    ax1.grid(ls=":")
    ax1.legend(loc="best", frameon=True, fontsize="large")

    # # precipitation
    # ax2.fill_between(x_axis, tp_plot, 0, color="blue", alpha=0.3)
    # ax2.plot(x_axis, tp_plot, color="blue", label="Total Precipitation")
    # ax2.set_ylabel("Precipitation (mm/6h)", fontsize="large")
    # ax2.grid(ls=":")

    # temperature
    ax3.plot(x_axis, T_s_plot, color="black", label="Surface Temperature ($T_s$)")
    ax3.set_ylabel("Temperature (°C)", fontsize="large")
    ax3.grid(ls=":")
    ax3.set_xlabel("Time (Days)", fontsize="x-large")
    ax3.set_xlim(x_axis.min(), x_axis.max())

    # fig.suptitle(title, fontsize="x-large")
    # plt.tight_layout()
    plt.savefig(
        f"iterative_boosting_plots/atmospheric_profile_{title.replace(' ', '_')}.pdf", bbox_inches="tight"
    )
    plt.show()

In [21]:
def plot_iterative_boosting(
    tsss, title, era=None, time_of_day=[0, 6, 12, 18], save=None
):
    plt.style.use("seaborn-v0_8-whitegrid")
    fig, ax = plt.subplots(figsize=get_figsize(WIDTH_INSA, 1, 0.65))
    cmap = ncp.pal_sample("Rubidium", len(tsss))

    for phase, tss in enumerate(tsss):
        tss = tss.where(
            tss.valid_time.dt.hour.isin(time_of_day).compute(), drop=True
        ).compute()
        lead_times_hours = tss.step.values / np.timedelta64(1, "h")
        phase_color = cmap[phase]

        for i, n in enumerate(tss.number.values):
            ts = tss.sel(number=n)
            current_label = f"Phase {phase + 1}" if i == 0 else "_nolegend_"
            ax.plot(
                lead_times_hours,
                ts,
                color=phase_color,
                label=current_label,
            )
    ax.set_title(title)
    ax.set_ylabel("Temperature (°C)")
    ax.set_xlabel("Time (date)")
    ax.set_ylim((12,46))
    ax.legend()
    ax.tick_params(axis="both", which="major")
    fig.autofmt_xdate(rotation=45)
    plt.tight_layout()
    if save is not None:
        plt.savefig(f"iterative_boosting_plots/{save}.pdf", bbox_inches="tight")
    plt.show()

# Plots

In [22]:
print(dataset_results_reconstructed_R1.time.to_series().value_counts()*1/4.8)

time
2022-07-08    79.166667
2015-06-20    20.833333
Name: count, dtype: float64


In [23]:
print(dataset_results_reconstructed_R2.time.to_series().value_counts()*1/4.8)

time
2022-07-08    54.166667
2015-06-20    25.000000
2022-07-06    12.500000
2015-06-18     4.166667
2015-06-19     4.166667
Name: count, dtype: float64


In [31]:
paris_temp = dataset_results_reconstructed_R1["2t"].isel(values=paris_index) - 273.15
lead_times = paris_temp.step.values / np.timedelta64(1, "D")

paris_temp_noon = paris_temp.where(
            paris_temp.valid_time.dt.hour.isin([12]).compute(), drop=True
        ).compute()
lead_times = lead_times[paris_temp.valid_time.dt.hour.isin([12]).isel(number=0).compute()]

plt.figure(figsize=get_figsize(WIDTH_INSA, 0.5, 0.6))

# median, and outer bounds
temp_median = paris_temp_noon.quantile(0.5, dim="number")
temp_95 = paris_temp_noon.quantile(0.95, dim="number")
temp_05 = paris_temp_noon.quantile(0.05, dim="number")

# 90% confidence interval
plt.fill_between(
    lead_times,
    temp_05,
    temp_95,
    color="blue",
    alpha=0.3,
    label="5th - 95th Percentile",
)

# Plot the masked data
plt.plot(lead_times, paris_temp_noon.values.T, color="black", alpha=0.1)

# ensemble median
plt.ylim((15,46))
plt.plot(lead_times, temp_median, color="red", linewidth=2, label="Ensemble Median")

plt.xlabel("Lead Time (days)")
plt.ylabel("Temperature (°C)")
plt.grid(True, linestyle="--", alpha=0.6)

plt.savefig(
    "iterative_boosting_plots/reconstructed_paris_R1.pdf", bbox_inches="tight"
)
plt.show()

In [32]:
paris_temp = dataset_results_reconstructed_R2["2t"].isel(values=paris_index) - 273.15
lead_times = paris_temp.step.values / np.timedelta64(1, "D")

paris_temp_noon = paris_temp.where(
            paris_temp.valid_time.dt.hour.isin([12]).compute(), drop=True
        ).compute()
lead_times = lead_times[paris_temp.valid_time.dt.hour.isin([12]).isel(number=0).compute()]

plt.figure(figsize=get_figsize(WIDTH_INSA, 0.5, 0.6))

# median, and outer bounds
temp_median = paris_temp_noon.quantile(0.5, dim="number")
temp_95 = paris_temp_noon.quantile(0.95, dim="number")
temp_05 = paris_temp_noon.quantile(0.05, dim="number")

# 90% confidence interval
plt.fill_between(
    lead_times,
    temp_05,
    temp_95,
    color="blue",
    alpha=0.3,
    label="5th - 95th Percentile",
)

# Plot the masked data
plt.plot(lead_times, paris_temp_noon.values.T, color="black", alpha=0.1)

# ensemble median
plt.ylim((15,46))
plt.plot(lead_times, temp_median, color="red", linewidth=2, label="Ensemble Median")

plt.xlabel("Lead Time (days)")
plt.ylabel("Temperature (°C)")
plt.grid(True, linestyle="--", alpha=0.6)

plt.savefig(
    "iterative_boosting_plots/reconstructed_paris_R2.pdf", bbox_inches="tight"
)
plt.show()

# Return times

## Detrending the historical data

In [ ]:
era5_paris_12utc_JJA = (
    era5_paris_ts.isel(
        valid_time=era5_paris_ts.valid_time.dt.month.isin([6, 7, 8])
        * era5_paris_ts.valid_time.dt.hour.isin([12])
    ).t2m
    - 273.15
)
era5_paris_12utc_JJA_max_per_year = era5_paris_12utc_JJA.groupby(
    "valid_time.year"
).max()

In [ ]:
EU_DATA = (
    xr.open_dataset("gistemp1200_GHCNv4_ERSSTv5.nc.gz")
    .rename_vars({"tempanomaly": "t2m"})
    .t2m
)
EU_DATA = (
    EU_DATA.sel(lat=slice(30, 70), lon=slice(-20, 50))
    .mean(["lat", "lon"])
    .groupby("time.year")
    .mean()
    .rolling(year=15, center=True)
    .mean()
    .interpolate_na(dim="year", method="linear", fill_value="extrapolate")
)
europe_temp_per_year = EU_DATA.sel(year=EU_DATA.year >= 1940)

In [ ]:
slope_europe, _ = np.polyfit(
    europe_temp_per_year, era5_paris_12utc_JJA_max_per_year, deg=1
)
gmst_2026_europe = europe_temp_per_year.isel(year=-1)

In [ ]:
era5_paris_12utc_JJA_max_per_year_detrended_europe = (
    era5_paris_12utc_JJA_max_per_year
    + slope_europe * (gmst_2026_europe - europe_temp_per_year)
)
era5_paris_12utc_JJA_detrended_europe = era5_paris_12utc_JJA + slope_europe * (
    gmst_2026_europe
    - europe_temp_per_year.sel(year=era5_paris_12utc_JJA.valid_time.dt.year)
)

## Finding historical extremes

In [ ]:
detrended_maxes = era5_paris_12utc_JJA_detrended_europe.groupby("valid_time.year").max()
peak_dates = [
    year_data.idxmax(dim="valid_time").values
    for year, year_data in era5_paris_12utc_JJA_detrended_europe.groupby(
        "valid_time.year"
    )
]
df_peaks = pd.DataFrame(
    {"Detrended_Max_Temp_C": detrended_maxes.values, "Exact_Date": peak_dates},
    index=detrended_maxes.year.values,
)
df_peaks.index.name = "Year"
df_peaks_sorted = df_peaks.sort_values(by="Detrended_Max_Temp_C", ascending=False)

print(df_peaks_sorted.head(15))

and 24 juin 2026

## Functions for return times

In [ ]:
def plot_return_times(model, return_periods, T_ext_array, ci_lower, ci_upper, name):
    fig, ax = plt.subplots(figsize=get_figsize(WIDTH_INSA,1,0.6))

    # plot historical return times + gev + confint
    obs_temp = np.sort(model.extremes.values)
    obs_rp = (len(obs_temp) + 1.0) / np.arange(len(obs_temp), 0, -1)
    gev_rp = np.logspace(0, 4, 100)
    gev_temp, gev_lower, gev_upper = model.get_return_value(gev_rp, alpha=0.95)

    ax.scatter(
        obs_rp,
        obs_temp,
        color="black",
        marker="o",
        label="Historical Data (ERA5)",
        alpha=0.75,
    )
    ax.plot(gev_rp, gev_temp, color="#F85C50", label="GEV Fit on historical data")
    ax.fill_between(
        gev_rp, gev_lower, gev_upper, color="#5199FF", alpha=0.25, label="95% CI (GEV)"
    )
    ax.fill_betweenx(T_ext_array, ci_lower, ci_upper, color="green", alpha=0.3,
                 label='95% bootstrap CI (Iterative Boosting)')
    # plot boosted estimated return times
    ax.plot(
        return_periods,
        T_ext_array,
        color="black",
        label="Iterative Boosting + AIFS-Ens",
    )
    
    # nicer plot
    ax.set_xscale("log")
    ax.set_xlim(1, 1e4)
    ax.set_ylim(28, 48)
    ax.set_xlabel("Estimated return period (Years)")
    ax.set_ylabel("Maximum temperature ($^\circ$C)")
    ax.grid(True, which="major", ls="-", alpha=0.6)
    ax.grid(True, which="minor", ls=":", alpha=0.4)
    ax.set_yticks(np.arange(28, 49, 2))
    ax.set_yticks(np.arange(28, 49, 1), minor=True)
    ax.legend(loc="lower right")
    plt.tight_layout()

    plt.savefig(f"return_times/return_periods_{name}.pdf", bbox_inches="tight")
    plt.show()

In [ ]:
def get_return_periods_from_json(
    json_path, T_ref_0, P_clim_T_ref_0, q=0.05, num_points=200, J=None, n_bootstraps=500
):
    # load json
    with open(json_path, "r") as f:
        phases = json.load(f)["phases"]
    if J is not None:
        phases = phases[:J]
    else:
        J = len(phases)

    # extract max temps and thresholds for all phases
    maxes = [np.array([m["max_temp"] for m in p["members"]]) - 273.15 for p in phases]
    t_refs = [T_ref_0] + [p["T_ref"] - 273.15 for p in phases[:-1]]

    # denominator
    denoms = [np.mean(m >= t) for m, t in zip(maxes, t_refs)]
    chained_denom = np.prod(denoms)

    for i, m in enumerate(maxes, start=1):
        phase_max = np.max(m)
        top_5_percent = np.percentile(m, 95)
        print(f"Phase {i}" f"Max: {phase_max:.2f}°C | Top 5%: {top_5_percent:.2f}°C")

    # T_ext array
    T_ext_array = np.linspace(t_refs[-1], np.max(maxes[-1]), num_points)

    return_periods = []
    q_factor = q ** (J - 1)

    for T_ext in T_ext_array:
        P_T_ext_AC = np.sum(maxes[-1] >= T_ext) / len(maxes[-1])
        P_T_ext_uncond = P_clim_T_ref_0 * (q_factor / chained_denom) * P_T_ext_AC
        if P_T_ext_uncond > 0:
            return_periods.append(1.0 / P_T_ext_uncond)
        else:
            return_periods.append(np.nan)

    all_boot_rps = []
    for _ in range(n_bootstraps):
        boot_maxes = [np.random.choice(m, size=len(m), replace=True) for m in maxes]

        boot_denoms = [np.mean(m >= t) for m, t in zip(boot_maxes, t_refs)]
        boot_chained_denom = np.prod(boot_denoms)

        boot_rp = []
        for T_ext in T_ext_array:
            P_T_ext_AC = np.sum(boot_maxes[-1] >= T_ext) / len(boot_maxes[-1])
            P_T_ext_uncond = (
                P_clim_T_ref_0 * (q_factor / boot_chained_denom) * P_T_ext_AC
            )
            if P_T_ext_uncond > 0:
                boot_rp.append(1.0 / P_T_ext_uncond)
            else:
                boot_rp.append(np.nan)

        all_boot_rps.append(boot_rp)

    ci_lower = np.nanpercentile(all_boot_rps, 2.5, axis=0)
    ci_upper = np.nanpercentile(all_boot_rps, 97.5, axis=0)

    return return_periods, T_ext_array, ci_lower, ci_upper


def get_gev(ts):
    # GEV
    model = EVA(data=ts)
    model.get_extremes(method="BM", extremes_type="high", block_size="365.2425D")
    model.fit_model(model="Emcee")

    return model

## Compute and plot return times

In [ ]:
GEV = get_gev(era5_paris_12utc_JJA_detrended_europe.squeeze().to_series())

In [ ]:
N_PARENT=16
N_HISTORICAL=len(era5_paris_12utc_JJA_max_per_year)

In [ ]:
return_periods, T_ext_array, ci_lower, ci_upper = get_return_periods_from_json(
    json_path="/scratchx/pchevali/ITERATIVE_BOOSTING_PROCESSED/return_levels_JJA_run2/return_levels_JJA_run2_lineage.json",
    T_ref_0=37.171132,
    P_clim_T_ref_0=N_PARENT / N_HISTORICAL,
    q=0.05,
    J=3,
    n_bootstraps=100,
)

In [ ]:
plot_return_times(GEV, return_periods, T_ext_array, ci_lower, ci_upper, name="iterative_boosting_JJA_R2")

In [ ]:
return_periods, T_ext_array, ci_lower, ci_upper = get_return_periods_from_json(
    json_path="/scratchx/pchevali/ITERATIVE_BOOSTING_PROCESSED/return_levels_JJA_run1/return_levels_JJA_run1_lineage.json",
    T_ref_0=37.171132,
    P_clim_T_ref_0=N_PARENT / N_HISTORICAL,
    q=0.05,
    J=3,
    n_bootstraps=100,
)

In [ ]:
plot_return_times(GEV, return_periods, T_ext_array, ci_lower, ci_upper, name="iterative_boosting_JJA_R1")

# Looking into really hot events

In [34]:
def temperature_decomposition(v, region, target_member):
    # Target pressure levels
    levels = [1000, 925, 850, 700, 600, 500, 400, 300, 250]
    # pressur levels in pascal
    levels_pa = [lvl * 100 for lvl in levels]

    # load what we'll use so it's faster
    v = (
        v[[f"{var}_{lvl}" for lvl in levels for var in ["t", "u", "v", "w"]]]
        .sel(number=target_member)
        .isel(values=region)
        .load()
    )

    # stack temperature fields cos they're the only ones we'll differentiate vertically
    T_stacked = xr.concat(
        [v[f"t_{lvl}"] for lvl in levels], dim=pd.Index(levels_pa, name="level")
    )

    # differentiation wrt time and pressure levels
    dT_dp_stacked = T_stacked.differentiate("level")
    dT_dt_stacked = (
        T_stacked.differentiate("step") * 1e9
    )  # step is in timedelta64[ns], differentiating gives K/ns --> multiply by 1e9 for K/s

    # fixed constants/values
    start_step = 0  # 1
    num_steps = 60  # 59
    end_step = start_step + num_steps
    times = np.arange(start_step, end_step)
    SEC_PER_DAY = 86400
    KAPPA = 0.286

    lon_1d = LON[region]
    lat_1d = LAT[region]

    # compute the grid parameters with random data
    grid_lon, grid_lat, _ = interpolate_to_grid(
        lon_1d,
        lat_1d,
        v["t_850"].isel(step=start_step).values,
        interp_type="linear",
        hres=0.25,
    )
    dx, dy = mpcalc.lat_lon_grid_deltas(grid_lon, grid_lat)

    # empty results matrixes
    shape = (len(levels), len(times))
    adv = np.zeros(shape)
    comp = np.zeros(shape)
    disp = np.zeros(shape)
    Qdiab = np.zeros(shape)
    dTdt = np.zeros(shape)
    T = np.zeros(shape)

    # COMPUTATION LOOP
    for t_idx, current_step in enumerate(times):
        v_step = v.isel(step=current_step)
        dT_dp_step = dT_dp_stacked.isel(step=current_step)
        dT_dt_step = dT_dt_stacked.isel(step=current_step)

        for l_idx, lvl in enumerate(levels):
            # extract 1D data for current level and step
            T_1d = v_step[f"t_{lvl}"].values
            u_1d = v_step[f"u_{lvl}"].values
            v_1d = v_step[f"v_{lvl}"].values
            w_1d = v_step[f"w_{lvl}"].values
            dT_dp_1d = dT_dp_step.sel(level=lvl * 100).values
            dT_dt_1d = dT_dt_step.sel(level=lvl * 100).values

            # interp to 2d for advection (first values returned are the grid so we don't gaf)
            _, _, T_2d = interpolate_to_grid(
                lon_1d, lat_1d, T_1d, interp_type="linear", hres=0.25
            )
            _, _, u_2d = interpolate_to_grid(
                lon_1d, lat_1d, u_1d, interp_type="linear", hres=0.25
            )
            _, _, v_2d = interpolate_to_grid(
                lon_1d, lat_1d, v_1d, interp_type="linear", hres=0.25
            )

            # compute advection using metpy
            adv_rate = mpcalc.advection(
                T_2d * units.kelvin,
                u=u_2d * units("m/s"),
                v=v_2d * units("m/s"),
                dx=dx,
                dy=dy,
            ).m

            # compute compression, vertical displascement and Q_diab terms
            comp_rate = w_1d * (KAPPA * T_1d / (lvl * 100))
            disp_rate = -1 * w_1d * dT_dp_1d

            # first spatial average so we can compute Q_diab too
            adv_rate_m = np.nanmean(adv_rate)
            comp_rate_m = np.mean(comp_rate)
            disp_rate_m = np.mean(disp_rate)
            dT_dt_1d_m = np.mean(dT_dt_1d)

            Qdiab_rate = dT_dt_1d_m - adv_rate_m - comp_rate_m - disp_rate_m
            # not really something we can compute so isolating it

            # scale to K/days
            adv[l_idx, t_idx] = adv_rate_m * SEC_PER_DAY
            comp[l_idx, t_idx] = comp_rate_m * SEC_PER_DAY
            disp[l_idx, t_idx] = disp_rate_m * SEC_PER_DAY
            Qdiab[l_idx, t_idx] = Qdiab_rate * SEC_PER_DAY
            dTdt[l_idx, t_idx] = dT_dt_1d_m * SEC_PER_DAY
            T[l_idx, t_idx] = np.mean(T_1d)

    # combined vertical term
    vert = comp + disp

    return (
        T,
        dTdt,
        adv,
        Qdiab,
        comp,
        disp,
        vert,
        v.valid_time.isel(step=times).values,
        levels,
    )

In [35]:
from scipy.ndimage import uniform_filter1d


def plot_vertical_decomposition(dTdt, adv, vert, T, plot_dates, levels, region, name):

    window = 4
    T_anom = T - np.tile(
        climato_era5_ta_interpolated.isel(values=region)
        .mean(dim="values")
        .sel(level=levels)
        .transpose("level", "hour")
        .values,
        reps=(1, 15),
    )
    T_anom = uniform_filter1d(T_anom, size=window, axis=1)
    dTdt = uniform_filter1d(dTdt, size=window, axis=1)
    adv = uniform_filter1d(adv, size=window, axis=1)
    vert = uniform_filter1d(vert, size=window, axis=1)

    lead_times = np.arange(T.shape[1]) * 0.25

    plot_data = [
        (T_anom, "(a)"),
        (dTdt, "(b)"),
        (adv, "(c)"),
        (vert,"(d)"),
    ]

    fig, axes = plt.subplots(
        nrows=2,
        ncols=2,
        figsize=get_figsize(WIDTH_INSA, 1, 0.65),
    )
    axes = axes.flatten()

    for idx, (ax, (data, title)) in enumerate(zip(axes, plot_data)):
        plot_levels = np.linspace(-15, 15, 16)

        cf = ax.contourf(
            lead_times,
            levels,
            data,
            levels=plot_levels,
            cmap="RdBu_r",
            extend="both",
        )

        if title == "(a)":
            fig.colorbar(
                cf,
                ax=ax,
                pad=0.02,
                label="Temperature anomaly($^\circ$C)",
                ticks=mtickers.MaxNLocator(integer=True),
            )
        else:
            fig.colorbar(
                cf,
                ax=ax,
                pad=0.02,
                label="Heating (K/day)",
                ticks=mtickers.MaxNLocator(integer=True),
            )

        if idx % 2 == 0:
            ax.set_ylabel("Pressure level (hPa)")
            ax.tick_params(labelleft=True)
        else:
            ax.tick_params(labelleft=False)

        if idx >= 2:
            ax.set_xlabel("Lead Time (days)")
            ax.tick_params(labelbottom=True)
        else:
            ax.tick_params(labelbottom=False)

        ax.set_title(title)
        ax.tick_params(axis="both", bottom=True, left=True, length=2)
        ax.tick_params(axis="x", rotation=45)
        ax.invert_yaxis()

    plt.tight_layout()

    plt.savefig(f"iterative_boosting_plots/hovmoller_{name}.pdf", bbox_inches="tight")
    plt.show()

In [36]:
def get_ith_peak_day_info(dataset, i, index):
    ds_loc = dataset.isel(values=index)
    maxes = ds_loc["2t"].max(dim="step").values

    ith_best_idx = np.argsort(maxes)[::-1][i]
    best_number = ds_loc["number"].values[ith_best_idx]
    ds_best_event = ds_loc.isel(number=ith_best_idx)

    peak_step_idx = ds_best_event["2t"].argmax(dim="step").compute().item()
    peak_valid_time = pd.to_datetime(ds_best_event["valid_time"].values[peak_step_idx])
    peak_date = peak_valid_time.date()

    event_valid_times = pd.to_datetime(ds_best_event["valid_time"].values)
    day_mask = event_valid_times.date == peak_date
    steps_for_day = ds_best_event["step"].values[day_mask]

    return best_number, steps_for_day

In [37]:
def plot_daily_z500_t12z(ds, number, steps, lon, lat, region):
    day_data = ds.sel(number=number, step=steps).isel(values=region)

    z500_daily = day_data["z_500"].mean(dim="step").values / 9.81

    hours = pd.to_datetime(day_data.valid_time.values).hour
    idx_12z = np.where(hours == 12)[0][0]
    t2m_12z = day_data["2t"].isel(step=idx_12z).values - climato_era5[region]

    fig = plt.figure(figsize=get_figsize(WIDTH_INSA, fraction=0.5, height_factor=1))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

    vmax = np.abs(t2m_12z).max()
    cf = ax.tricontourf(
        lon[region], lat[region], t2m_12z, levels=15, cmap=ncp.pal("vanadium"), vmin=-vmax, vmax=vmax
    )
    cbar = fig.colorbar(cf, ax=ax, orientation="horizontal", shrink=0.6, pad=0.01)
    cbar.set_label("Temperature anomaly at 12 UTC ($^\circ$C)")

    cs = ax.tricontour(lon[region], lat[region], z500_daily, levels=14, colors="black", linewidths=1)
    ax.clabel(cs, fmt="%.0f", inline=True)

    ax.coastlines(linewidth=0.3)
    plt.savefig(f"iterative_boosting_plots/map_hottest_day{number}.pdf", bbox_inches="tight")
    plt.show()

In [ ]:
for i in range(10):

    best_number, steps_for_day = get_ith_peak_day_info(
        dataset_results_reconstructed_R1, i, paris_index
    )

    print(
        f"Member {best_number}, started on : {dataset_results_reconstructed_R1.sel(number=best_number).time.values}"
    )

    plot_daily_z500_t12z(
        dataset_results_reconstructed_R1,
        best_number,
        steps_for_day,
        dataset_results_reconstructed_R1.longitude.values,
        dataset_results_reconstructed_R1.latitude.values,
        plot_region_indexs,
    )
    MSE_compute_and_plot(
        dataset_results_reconstructed_R1.sel(number=best_number),
        region=[paris_index],
        title=f"run1_member_{best_number}",
    )

    (
        T_hot,
        dTdt_hot,
        adv_hot,
        Qdiab_hot,
        comp_hot,
        disp_hot,
        vert_hot,
        times_hot,
        levels_hot,
    ) = temperature_decomposition(
        v=dataset_results_reconstructed_R1,
        region=paris_big_box_region,
        target_member=best_number,
    )

    plot_vertical_decomposition(
        dTdt_hot,
        adv_hot,
        vert_hot,
        T_hot,
        times_hot,
        levels_hot,
        paris_big_box_region,
        f"run1_member_{best_number}",
    )

In [38]:
n_events = 10

T_comp, dTdt_comp, adv_comp, vert_comp = 0, 0, 0, 0
times_hot, levels_hot = None, None

for i in range(n_events):
    best_number, steps_for_day = get_ith_peak_day_info(
        dataset_results_reconstructed_R1, i, paris_index
    )
    print(
        f"Member {best_number}, started on : {dataset_results_reconstructed_R1.sel(number=best_number).time.values}"
    )

    (
        T_hot,
        dTdt_hot,
        adv_hot,
        Qdiab_hot,
        comp_hot,
        disp_hot,
        vert_hot,
        times,
        levels,
    ) = temperature_decomposition(
        v=dataset_results_reconstructed_R1,
        region=paris_big_box_region,
        target_member=best_number,
    )

    T_comp += T_hot / n_events
    dTdt_comp += dTdt_hot / n_events
    adv_comp += adv_hot / n_events
    vert_comp += vert_hot / n_events

    times_hot = times
    levels_hot = levels

Member 5, started on : 2015-06-20T00:00:00.000000
Member 199, started on : 2022-07-08T00:00:00.000000
Member 113, started on : 2022-07-08T00:00:00.000000
Member 265, started on : 2022-07-08T00:00:00.000000
Member 144, started on : 2022-07-08T00:00:00.000000
Member 287, started on : 2015-06-20T00:00:00.000000
Member 148, started on : 2022-07-08T00:00:00.000000
Member 108, started on : 2022-07-08T00:00:00.000000
Member 145, started on : 2022-07-08T00:00:00.000000
Member 58, started on : 2022-07-08T00:00:00.000000


In [39]:
plot_vertical_decomposition(
    dTdt_comp,
    adv_comp,
    vert_comp,
    T_comp,
    times_hot,
    levels_hot,
    paris_big_box_region,
    "composite_all_events",
)